# Audio Dataset Preparation and Generation

This notebook prepares and generates training data for the RAVE model.

The unprocessed data is found in ``rave/data/unprepared_data`` (unzip the files).
- it contains **bird&nature** data (``data/unprepared_data/kaggle_birds_data``) and **song** data (``data/unprepared_data/song_data``)
- the songs are all by Cosmo Sheldrake. The song "Nightjar" is also contained in of the folder.

The notebook does the following:
- It processes bird recordings and song files into mono .wav files (recommended for RAVE)
- The audio files are trimmed, standardized, and saved into a folder
- All WAV files are then checked, normalized, and converted to a shared sample rate

## 1. Imports and global settings

In [ ]:
import random
from pathlib import Path
import torch
from pydub import AudioSegment
import soundfile as sf
import numpy as np
import shutil
from math import gcd
from scipy.signal import resample_poly

torch.set_grad_enabled(False)

## 2. Configuration

In [ ]:
# Step 1: bird dataset preparation
BIRD_INPUT_DIR = Path("data/unprepared_data/kaggle_birds_data")
BIRD_OUTPUT_DIR = Path("data/unprepared_data/intermediate_data/birds")
MAX_FILES_PER_CLASS = 3
BIRD_MAX_DURATION_MS = 20_000  # 20 seconds
BIRD_SEED = 42

# Step 2: song dataset preparation
SONG_INPUT_DIR = Path("data/unprepared_data/song_data")
SONG_OUTPUT_DIR = Path("data/unprepared_data/intermediate_data/songs")
SNIPPET_MS = 20_000  # 20 seconds per window
OFFSETS_MS = [0, 5_000, 10_000]  # 0s, 5s, and 10s starting offsets
MIN_SNIPPET_MS = 5_000  # discard tail snippets shorter than 5 seconds
SUPPORTED_EXT = {".mp3", ".wav"}

# Step 3: sample-rate unification and verification
UNIFY_INPUT_DIR = Path("data/unprepared_data/intermediate_data")
TRAINING_OUTPUT_DIR = Path("data/training_data")
TARGET_RMS = 0.1

## 3. Step 1 — prepare bird dataset

- Find species folders under `data/unprepared_data/kaggle_birds_data`
- Randomly select up to 3 `.mp3` files per species (seed 42)
- Trim files longer than 20 seconds
- Convert to mono
- Export WAV files under `data/unprepared_data/intermediate_data/birds`

In [ ]:
random.seed(BIRD_SEED)
BIRD_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

species_dirs = [d for d in BIRD_INPUT_DIR.iterdir() if d.is_dir()]

total_saved = 0

for species_dir in sorted(species_dirs):
    files = list(species_dir.glob("*.mp3"))

    # Shuffle the files so the selection is random (seed is used)
    random.shuffle(files)
    selected = files[:MAX_FILES_PER_CLASS]

    for i, src in enumerate(selected):
        out_path = BIRD_OUTPUT_DIR / f"{species_dir.name}_{i}.wav"

        # Load the source audio file
        audio = AudioSegment.from_file(src)

        # Trim only if the file is longer than the maximum duration
        # Short files stay at their original length
        if len(audio) > BIRD_MAX_DURATION_MS:
            audio = audio[:BIRD_MAX_DURATION_MS]

        # Convert the audio to mono
        audio = audio.set_channels(1)

        # Save the processed file as a WAV file
        audio.export(out_path, format="wav")

        print(f"[OK]   {out_path.name}")
        total_saved += 1

print(f"\nDone. Saved {total_saved} files.")

## 4. Step 2 — prepare song dataset

- Read `.mp3` and `.wav` files from `data/unprepared_data/song_data`
- Make three passes per song: offset 0 seconds, 5 seconds, and 10 seconds
- Slice consecutive 20-second windows from each offset
- Discard tail snippets shorter than 5 seconds
- Convert snippets to mono and export WAV files under `data/unprepared_data/intermediate_data/songs`

In [ ]:
SONG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

src_files = [
    f for f in SONG_INPUT_DIR.iterdir()
    if f.suffix.lower() in SUPPORTED_EXT
]

total_saved = 0

for src in sorted(src_files):
    print(f"\nProcessing: {src.name}")
    
    # Load the song file.
    audio = AudioSegment.from_file(src)

    base = src.stem

    for offset_ms in OFFSETS_MS:
        snippets = []

        # Start slicing at the current offset
        start = offset_ms

        while start < len(audio):
            # Take a window of SNIPPET_MS milliseconds
            chunk = audio[start : start + SNIPPET_MS]

            # Stop if the remaining tail is too short
            if len(chunk) < MIN_SNIPPET_MS:
                break

            # Convert the snippet to mono and keep it
            snippets.append(chunk.set_channels(1))

            # Move to the next non-overlapping window
            start += SNIPPET_MS

        for i, snippet in enumerate(snippets):
            offset_s = offset_ms // 1000
            out_name = f"{base}_off{offset_s}s_part{i}.wav"
            out_path = SONG_OUTPUT_DIR / out_name

            # Save the snippet as a WAV file
            snippet.export(out_path, format="wav")

            print(f"[OK]  {out_name}")
            total_saved += 1

print(f"\nDone. Saved {total_saved} snippets total.")

## 5. Step 3 — unify sample rate and verify dataset

- Scan all WAV files created in the previous preparation steps
- resample to 44100 Hz if needed
- normalize volume with RMS normalization
- save the cleaned files into the final training dataset folder
- verify that the output files are consistent

In [ ]:
# Unify the prepared audio dataset:
# - scan all WAV files created in the previous preparation steps
# - resample to 44100 Hz if needed
# - normalize volume with RMS normalization to 0.1
# - save the cleaned files into the final training dataset folder
# - verify that the output files are consistent

target_sr = 44100
target_rms = 0.1

# Remove old output files so reruns do not mix old and new results
if TRAINING_OUTPUT_DIR.exists():
    shutil.rmtree(TRAINING_OUTPUT_DIR)

TRAINING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

files = list(UNIFY_INPUT_DIR.rglob("*.wav"))

print("Scanning prepared_data …")
print(f"Found {len(files)} WAV files.")
print(f"Target sample rate: {target_sr} Hz")
print()

saved = 0

for src in files:
    rel = src.relative_to(UNIFY_INPUT_DIR)
    dst = TRAINING_OUTPUT_DIR / rel

    # Load audio
    wav, sr = sf.read(str(src), dtype="float32", always_2d=True)

    # soundfile loads as (samples, channels), so convert to (samples,)
    wav = wav[:, 0]

    # remove DC offset
    wav = wav - wav.mean()

    # resample to 44100 Htz
    if sr != target_sr:
        factor = gcd(sr, target_sr)
        wav = resample_poly(
            wav,
            target_sr // factor,
            sr // factor
        ).astype(np.float32)

    # normalize RMS
    rms = np.sqrt(np.mean(wav**2))

    if rms > 1e-8:
        wav = wav * (target_rms / rms)

    wav = np.clip(wav, -1.0, 1.0)

    dst.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(dst), wav, target_sr, subtype="FLOAT")

    print(f"[OK] {rel}")
    saved += 1

print("Done.")